In [1]:
import os, json, re, html, unicodedata
from pathlib import Path
import pandas as pd
from urllib.parse import urlparse

import sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "../..")))
import config

fandom_name = urlparse(config.BASE_URL).netloc.split(".")[0]
RAW_DATA_DIR = Path(config.FANDOM_DATA_DIR)
DATASET_DIR  = Path(config.BASE_DIR).parents[1] / "9.Span_Identification" / "datasets" / "processed"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

INPUT = RAW_DATA_DIR / f"master_spans_{fandom_name}.csv"
print("INPUT:", INPUT)
print("OUT  :", DATASET_DIR)

INPUT: /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/master_spans_alldimensions.csv
OUT  : /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed


In [49]:
usecols = ["article_id","paragraph_id","paragraph_text","start","end","link_text"]
df = pd.read_csv(INPUT, usecols=usecols).rename(columns={"paragraph_text":"text"})
print(len(df), df.columns.tolist())

315635 ['article_id', 'paragraph_id', 'text', 'link_text', 'start', 'end']


In [50]:
df["text"] = df["text"].fillna("").astype(str)
df["link_text"] = df["link_text"].fillna("").astype(str)
df["start"] = df["start"].astype(int)
df["end"]   = df["end"].astype(int)

In [51]:
def norm(s):
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s)
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_slice(t, s, e):
    n = len(t); s = max(0, min(s, n)); e = max(0, min(e, n))
    return t[s:e] if e > s else ""

In [52]:
def try_fix_span(t, s, e, lt):
    t_strip = safe_slice(t, s, e).strip()
    if t_strip == lt.strip(): 
        return s, e
    n = len(t)
    for ds in (-2,-1,0,1,2):
        for de in (-2,-1,0,1,2):
            ss, ee = max(0, s+ds), min(n, e+de)
            if ss < ee and t[ss:ee].strip() == lt.strip():
                return ss, ee
    # also try normalized match once
    lt_n = norm(lt)
    for ds in (-2,-1,0,1,2):
        for de in (-2,-1,0,1,2):
            ss, ee = max(0, s+ds), min(n, e+de)
            if ss < ee and norm(t[ss:ee]) == lt_n:
                return ss, ee
    return s, e

In [53]:
df[["start","end"]] = [
    try_fix_span(t, s, e, lt)
    for t, s, e, lt in zip(df["text"], df["start"], df["end"], df["link_text"])
]

In [54]:
slice_now = [safe_slice(t, s, e) for t,s,e in zip(df["text"], df["start"], df["end"])]
mismatch = sum(a.strip() != b.strip() for a,b in zip(slice_now, df["link_text"]))
print("mismatches after fix:", mismatch)

mismatches after fix: 331


In [55]:
grp = (df.groupby(["article_id","paragraph_id","text"], as_index=False)
         .agg({"start": list, "end": list}))

grp["spans"] = [[ [a,b] for a,b in zip(st, en) ] for st,en in zip(grp["start"], grp["end"])]
grp = grp.drop(columns=["start","end"])
len(grp), grp.head(2)

(28490,
    article_id  paragraph_id  \
 0        2055            11   
 1        2055            15   
 
                                                 text                     spans  
 0  It is also the only zero-dimensional shape, an...  [[225, 230], [225, 230]]  
 1                                   vertex count = 1        [[0, 12], [0, 12]]  )

In [56]:
def validate_clip_dedup(text, spans):
    n = len(text); out = []
    for a,b in spans:
        if a > b: a,b = b,a
        a = max(0, min(int(a), n)); b = max(0, min(int(b), n))
        if b - a > 0: out.append((a,b))
    seen, uniq = set(), []
    for a,b in out:
        if (a,b) not in seen:
            seen.add((a,b)); uniq.append([a,b])
    return uniq

grp["spans"] = [validate_clip_dedup(t, s) for t,s in zip(grp["text"], grp["spans"])]

rows_with_any = (grp["spans"].str.len() > 0).sum()
total_spans   = sum(len(x) for x in grp["spans"])
print("rows:", len(grp), "| rows_with>=1:", rows_with_any, "| total_spans:", total_spans)

rows: 28490 | rows_with>=1: 28485 | total_spans: 157773


In [57]:
data = grp[grp["spans"].str.len() > 0].reset_index(drop=True)
data = data[["article_id","paragraph_id","text","spans"]]
print(len(data))

28485


In [58]:
from sklearn.model_selection import train_test_split
train_df, tmp_df = train_test_split(data, test_size=0.20, random_state=42, shuffle=True)
dev_df,   test_df = train_test_split(tmp_df, test_size=0.50, random_state=42, shuffle=True)
print("train/dev/test:", len(train_df), len(dev_df), len(test_df))

train/dev/test: 22788 2848 2849


In [59]:
def dumps_spans(lst): 
    return json.dumps(lst, ensure_ascii=False)

for name, part in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    out = DATASET_DIR / f"{name}.csv"
    p = part.copy()
    p["spans"] = p["spans"].map(dumps_spans)
    p.to_csv(out, index=False)
    print("saved:", out, "rows:", len(part))

saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/train.csv rows: 22788
saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/dev.csv rows: 2848
saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/test.csv rows: 2849


In [60]:
def dumps_spans(lst): 
    return json.dumps(lst, ensure_ascii=False)

for name, part in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    out = DATASET_DIR / f"{name}.csv"
    p = part.copy()
    p["spans"] = p["spans"].map(dumps_spans)
    p.to_csv(out, index=False)
    print("saved:", out, "rows:", len(part))

saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/train.csv rows: 22788
saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/dev.csv rows: 2848
saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed/test.csv rows: 2849


In [61]:
# schema
assert set(train_df.columns) == {"article_id","paragraph_id","text","spans"}
# sum check
assert len(train_df)+len(dev_df)+len(test_df) == len(data)
# reload one file
chk = pd.read_csv(DATASET_DIR / "train.csv")
assert chk["spans"].map(lambda s: isinstance(json.loads(s), list)).all()
print("✓ OK")

✓ OK


In [62]:
import json

# 1. Check that all splits add up
assert len(train_df) + len(dev_df) + len(test_df) == len(data), "Split mismatch!"

# 2. Schema check
expected_cols = {"article_id","paragraph_id","text","spans"}
for name in ["train","dev","test"]:
    path = DATASET_DIR / f"{name}.csv"
    df_check = pd.read_csv(path)
    assert set(df_check.columns) == expected_cols, f"{name}.csv has wrong columns"
    # 3. Ensure spans can be parsed back to list
    assert df_check["spans"].map(lambda s: isinstance(json.loads(s), list)).all(), f"{name}.csv spans not valid JSON"
    print(f"✓ {name}.csv looks good ({len(df_check)} rows)")

print("All sanity checks passed ✅")

✓ train.csv looks good (22788 rows)
✓ dev.csv looks good (2848 rows)
✓ test.csv looks good (2849 rows)
All sanity checks passed ✅
